# Learned positional embeddings

Цель: добавить к token embeddings информацию о позиции и понять broadcasting `(T, D) → (B, T, D)`.

## 1. Разминка

1. Почему token embeddings не содержат порядок токенов?
2. Чем фактический `sequence_length` отличается от максимального context window?
3. Почему token и positional embeddings должны иметь одинаковый `D`?
4. Какие формы будут у `positions` и `position_vectors` при `T=3`, `D=4`?

**Мои ответы:**

1. Token embedding table сопоставляет каждому token ID одну строку и не зависит от позиции этого ID в последовательности. Порядок нужно передать модели отдельно.
2. `sequence_length=T` — фактическое число позиций в текущем входном тензоре. `max_context_length` — максимальное допустимое значение `T` для модели. Например, при максимуме 16 текущая последовательность может иметь длину 3.
3. Мы складываем token и positional embeddings поэлементно, поэтому оба вектора должны иметь одинаковую длину `D`.
4. `positions.shape == (3,)`, потому что это три целочисленных индекса `[0, 1, 2]`; после lookup `position_vectors.shape == (3, 4)`.

In [1]:
import torch
from torch import nn

torch.manual_seed(42)

## 2. Token embeddings

Используем две последовательности длиной 3. Один и тот же token ID на разных позициях сначала выбирает одну и ту же строку token embedding table.

In [4]:
token_ids = torch.tensor([
    [1, 2, 3],
    [3, 2, 1],
])

vocab_size = 5
d_model = 4
token_embedding = nn.Embedding(vocab_size, d_model)

# Получаем token vectors.
token_vectors = token_embedding(token_ids)

B, T = token_ids.shape
print("IDs:", token_ids.shape)
print("token vectors:", token_vectors.shape)

IDs: torch.Size([2, 3])
token vectors: torch.Size([2, 3, 4])


## 3. Номера позиций

Для каждой последовательности нужны позиции `0, 1, ..., T-1`. Они одинаковы для всех элементов batch, поэтому достаточно одного одномерного тензора.

In [5]:
# Создаём позиции от 0 до T-1.
positions = torch.arange(T)

print(positions)
print(positions.shape)

tensor([0, 1, 2])
torch.Size([3])


**Объясни:** почему `positions.shape == (T,)`, а не `(B, T)`?

Позиции `0, 1, ..., T-1` одинаковы для каждой последовательности batch. Их достаточно создать один раз, а при сложении PyTorch распространит positional vectors по batch dimension.

## 4. Learned position embedding table

Таблица содержит отдельный обучаемый вектор длины `D` для каждой допустимой позиции от `0` до `max_context_length-1`.

In [6]:
max_context_length = 16
position_embedding = nn.Embedding(max_context_length, d_model)

# Преобразуем номера позиций в positional vectors.
position_vectors = position_embedding(positions)

print("position table:", position_embedding.weight.shape)
print("positions:", positions.shape)
print("position vectors:", position_vectors.shape)

position table: torch.Size([16, 4])
positions: torch.Size([3])
position vectors: torch.Size([3, 4])


**Вопрос:** сколько параметров содержит position embedding table и почему не только `T × D`?

Таблица содержит `max_context_length × D = 16 × 4 = 64` параметра: по одному вектору для каждой допустимой позиции модели. `T × D = 3 × 4` — только выбранная для текущего входа часть таблицы; в другом batch фактический `T` может быть больше.

## 5. Сложение и broadcasting

Итоговый вход Transformer:

$$X_{b,t}=E[token\_id_{b,t}]+P[t]$$

In [7]:
# Складываем token и position vectors.
X = token_vectors+position_vectors

print("token vectors:", token_vectors.shape)
print("position vectors:", position_vectors.shape)
print("Transformer input:", X.shape)

token vectors: torch.Size([2, 3, 4])
position vectors: torch.Size([3, 4])
Transformer input: torch.Size([2, 3, 4])


**Объясни:** как PyTorch сложил `(B, T, D)` и `(T, D)`? Какой positional vector прибавился к `X[1, 2]`?

PyTorch выравнивает формы справа и мысленно добавляет к `(T, D)` ведущее измерение: `(1, T, D)`. Затем этот тензор распространяется по batch dimension до `(B, T, D)`. Поэтому к `X[1, 2]` прибавляется `position_vectors[2]`: номер последовательности `1` не меняет positional vector, а индекс позиции `2` выбирает `P[2]`.

## 6. Проверка роли позиции

ID `1` находится в первой последовательности на позиции 0 и во второй на позиции 2. Token vector одинаковый, но итоговые входные представления должны различаться.

In [8]:
print("same token vectors:", torch.equal(token_vectors[0, 0], token_vectors[1, 2]))
print("same final vectors:", torch.equal(X[0, 0], X[1, 2]))
print("token at position 0:", X[0, 0])
print("same token at position 2:", X[1, 2])

same token vectors: True
same final vectors: False
token at position 0: tensor([ 0.0883, -0.7657, -1.3881,  0.8845], grad_fn=<SelectBackward0>)
same token at position 2: tensor([ 0.2355, -1.1211,  3.5380,  1.7541], grad_fn=<SelectBackward0>)


**Наблюдение:** объясни результаты двух сравнений.

Одинаковые token IDs выбирают одну строку token embedding table, поэтому `token_vectors[0, 0]` и `token_vectors[1, 2]` равны. Но они находятся на позициях 0 и 2: к ним прибавляются разные векторы `P[0]` и `P[2]`, поэтому итоговые представления `X[0, 0]` и `X[1, 2]` различаются.

## 7. Ограничение learned positional embeddings

Попробуй только предсказать результат, не меняя основной эксперимент:

```python
too_long_positions = torch.arange(17)
position_embedding(too_long_positions)
```

Почему возникнет ошибка при `max_context_length=16`?

Таблица размера `max_context_length=16` содержит строки только с индексами `0–15` и поэтому поддерживает последовательность длиной не более 16. `torch.arange(17)` создаёт индексы `0–16`; обращение к отсутствующей строке с индексом `16` вызовет `IndexError: index out of range`.

## 8. Вопросы для собеседования

1. Зачем Transformer нужна positional information?
2. Как работают learned positional embeddings?
3. Каковы формы token vectors, position vectors и их суммы?
4. Почему position vectors можно создать без batch dimension?
5. Какие параметры фиксированы архитектурой, а какие могут меняться на inference?
6. Какое ограничение есть у таблицы learned positional embeddings?

**Мои ответы:**

1. Self-attention сам по себе не получает явной информации о порядке токенов. Positional information позволяет различать одинаковые токены на разных позициях и последовательности с разным порядком.
2. Learned positional embedding — обучаемая таблица формы `(max_context_length, D)`. Номер позиции используется как индекс строки, а выбранный вектор прибавляется к token embedding соответствующей позиции.
3. Token vectors имеют форму `(B, T, D)`, position vectors — `(T, D)` или явно `(1, T, D)`, а после broadcasting их сумма сохраняет форму `(B, T, D)`.
4. Номера позиций `0, ..., T-1` одинаковы для всех последовательностей batch. PyTorch распространяет один набор position vectors по batch dimension.
5. `d_model`, vocabulary size, число слоёв и heads, тип позиционного механизма и размер learned position table фиксируют формы параметров. Фактические `B` и `T` могут меняться на inference, но `T` не должен превышать поддерживаемый context window.
6. Таблица learned positional embeddings содержит только заранее созданное число строк. Позицию за пределами `max_context_length-1` нельзя использовать без изменения или расширения параметров модели; кроме того, необученные новые строки не получают полезных представлений автоматически.